In [1]:
from frameworks.LightenDiffusion.models import CTDN
from dataset_registery.registery import DatasetRegistry
from torchsummary import summary
from eda.helpers.training_helpers import train_model, get_optimizer
from eda.helpers.losses import ctdn_loss_wrapper
import torch
%load_ext autoreload
%autoreload 2

/home/grads/o/omarkhater/projects/lle-generative-priors/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_name = "NSICE_paired"
dataset_id = "okhater/SICE_paired"
source = "huggingface"

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device used : {device}")

Device used : cuda


In [7]:
registry = DatasetRegistry()
registry.register_dataset(
    name=data_name,
    dataset_id=dataset_id,
    splits=["train", "test"],
    dataset_type="paired",
    data_dir=f"../../datasets/{data_name}",
)


train_loader = registry.get_dataloader(data_name, "train", batch_size=32, shuffle=True)
test_loader = registry.get_dataloader(data_name, "test", batch_size=32, shuffle=True)

Generating train split: 604 examples [00:00, 15443.17 examples/s]
Generating test split: 116 examples [00:00, 11388.76 examples/s]


In [8]:
model = CTDN()
model = model.to(device)

In [9]:
summary(model, (3, 256, 256), batch_size=1)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1          [1, 64, 256, 256]           4,864
            Conv2d-2          [1, 64, 256, 256]         102,464
            Conv2d-3          [1, 64, 256, 256]          36,928
         LeakyReLU-4          [1, 64, 256, 256]               0
            Conv2d-5          [1, 64, 256, 256]          36,928
            Conv2d-6          [1, 64, 256, 256]           4,160
         Res_block-7          [1, 64, 256, 256]               0
            Conv2d-8          [1, 64, 128, 128]          36,928
            Conv2d-9         [1, 128, 128, 128]          73,856
        LeakyReLU-10         [1, 128, 128, 128]               0
           Conv2d-11         [1, 128, 128, 128]         147,584
           Conv2d-12         [1, 128, 128, 128]           8,320
        Res_block-13         [1, 128, 128, 128]               0
           Conv2d-14           [1, 128,

In [10]:
optimizer=get_optimizer(model)


In [11]:
model, losses = train_model(
        model,
        data_loaders=(train_loader, test_loader),
        criterion=ctdn_loss_wrapper, 
        optimizer=optimizer, 
        num_epochs = 100, 
        batch_size= 32, 
        device=device, 
        val_frequency = 5,
        patience = 5
        )

Epoch 1/100: 100%|██████████| 10/10 [00:53<00:00,  5.33s/batch, loss=0.167]


Epoch: 0, avg_val_loss = 0.17065156


Epoch 6/100: 100%|██████████| 10/10 [00:54<00:00,  5.48s/batch, loss=0.192]


Epoch: 5, avg_val_loss = 0.17366894


Epoch 11/100: 100%|██████████| 10/10 [00:54<00:00,  5.46s/batch, loss=0.185]


...

Epoch: 15, avg_val_loss = 0.17916181


Epoch 21/100: 100%|██████████| 10/10 [00:54<00:00,  5.45s/batch, loss=0.166]


Epoch: 20, avg_val_loss = 0.17880059


Epoch 26/100: 100%|██████████| 10/10 [00:53<00:00,  5.37s/batch, loss=0.183]


Epoch: 25, avg_val_loss = 0.17699545
Early stopping triggered at epoch 26
Best epoch: 1, Best loss: 0.17065155506134033
